<a href="https://colab.research.google.com/github/AkshayAJoseph/OSC-day3/blob/main/day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Day 3: Architecture Tuning, Exporting Models & Docker Deployment
**TRAIN.TUNE.COMPETE – AI & ML Workshop**  
*Open Source Club, Saintgits College of Engineering*

In this session, we transition from training models inside research notebooks to building and packaging a production-ready AI microservice.

### Step 1: Dependencies & Hardware Setup
First, let's make sure we have all necessary web and imaging libraries installed, and check our execution device.

In [1]:
!pip install -q fastapi uvicorn python-multipart httpx pillow torch torchvision

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from PIL import Image
import os
import io
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running execution environment on: {device}")

Running execution environment on: cpu


### Step 2: Architecture Selection (ResNet vs. MobileNetV3)
Not all architectures fit every hardware profile. While ResNet is great for server-side accuracy, **MobileNetV3** uses depthwise separable convolutions designed specifically for mobile devices, drones, and edge IoT devices.

Let's see how easy it is to configure a MobileNetV3 backbone:

In [2]:
# Load pre-trained MobileNetV3-Large
mobilenet = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)

# Freeze pre-trained feature extractor
for param in mobilenet.parameters():
    param.requires_grad = False

# Swap the classification head (in MobileNet, the layer is inside the 'classifier' container at index 3)
num_features = mobilenet.classifier[3].in_features
NUM_CLASSES = 3  # E.g., Healthy, Rust, Blight
mobilenet.classifier[3] = nn.Linear(num_features, NUM_CLASSES)

print("MobileNetV3 configured successfully:")
print(mobilenet.classifier)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 166MB/s]


MobileNetV3 configured successfully:
Sequential(
  (0): Linear(in_features=960, out_features=1280, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1280, out_features=3, bias=True)
)


### Step 3: Quick Model Training & Weight Serialization (.pth)
We will train our primary ResNet-18 model on sample data, then export **only the learned parameter state dictionary (`state_dict`)**.

In [3]:
# 1. Setup local dataset directory structure
classes = ["healthy", "rust", "blight"]
for split in ["train", "val"]:
    for cls in classes:
        os.makedirs(f"data/{split}/{cls}", exist_ok=True)

# Generate synthetic image samples
for cls_idx, cls in enumerate(classes):
    for i in range(20):
        color = (cls_idx * 80, 255 - (cls_idx * 50), 100 + (cls_idx * 40))
        img = Image.new("RGB", (224, 224), color=color)
        img.save(f"data/train/{cls}/img_{i}.jpg")
    for i in range(5):
        color = (cls_idx * 80, 255 - (cls_idx * 50), 100 + (cls_idx * 40))
        img = Image.new("RGB", (224, 224), color=color)
        img.save(f"data/val/{cls}/img_{i}.jpg")

# 2. DataLoaders
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
train_data = datasets.ImageFolder('data/train', transform=transform)
train_loader = DataLoader(train_data, batch_size=8, shuffle=True)

# 3. Train ResNet-18 Head
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, len(classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

model.train()
for epoch in range(2):
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# 4. Export the weights to disk
torch.save(model.state_dict(), "vision_model.pth")
print("Model weights successfully serialized and saved to 'vision_model.pth'!")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 109MB/s]


Model weights successfully serialized and saved to 'vision_model.pth'!


### Step 4: Building the FastAPI Inference Server (`app.py`)
We create a standalone script `app.py` that exposes HTTP routes (`/` for health checks and `/predict` for image classification).

In [4]:
%%writefile app.py
from fastapi import FastAPI, File, UploadFile
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import io

app = FastAPI(title="AI Vision Classifier API")

# Target class names
CLASSES = ["healthy", "rust", "blight"]

# Load architecture and weights on CPU for lightweight deployment
model = models.resnet18()
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
model.load_state_dict(torch.load("vision_model.pth", map_location="cpu"))
model.eval()

# Preprocessing pipeline
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

@app.get("/")
def health_check():
    return {"status": "online", "model": "ResNet-18 Vision Agent"}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")
    tensor = transform(image).unsqueeze(0)

    with torch.no_grad():
        outputs = model(tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]
        confidence, pred_idx = torch.max(probabilities, 0)

    return {
        "prediction": CLASSES[pred_idx.item()],
        "confidence": f"{confidence.item() * 100:.2f}%"
    }

Writing app.py


### Step 5: In-Memory API Endpoint Testing
We can verify that our server handles requests and multipart image file uploads directly using FastAPI's `TestClient`.

In [5]:
from fastapi.testclient import TestClient
import app as backend

client = TestClient(backend.app)

# 1. Test GET /
res_health = client.get("/")
print("GET / Response:", res_health.json())

# 2. Test POST /predict with a sample image
sample_img_path = "data/val/rust/img_0.jpg"
with open(sample_img_path, "rb") as f:
    res_predict = client.post(
        "/predict",
        files={"file": ("sample.jpg", f, "image/jpeg")}
    )

print("\nPOST /predict Response:")
print(json.dumps(res_predict.json(), indent=2))

GET / Response: {'status': 'online', 'model': 'ResNet-18 Vision Agent'}

POST /predict Response:
{
  "prediction": "blight",
  "confidence": "42.49%"
}


### Step 6: Creating the Production Dockerfile
To containerize this application, we write a `Dockerfile` that packages Python, our minimal dependencies, `app.py`, and `vision_model.pth`.

In [6]:
%%writefile Dockerfile
FROM python:3.10-slim

WORKDIR /app

# Install lightweight CPU-only PyTorch and API server dependencies
RUN pip install --no-cache-dir \
    torch torchvision --index-url https://download.pytorch.org/whl/cpu \
    fastapi uvicorn pillow python-multipart

# Copy application code and exported weights
COPY app.py vision_model.pth ./

EXPOSE 8000

# Run Uvicorn production server
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

Writing Dockerfile
